# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">Rewriting</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know how to implement query rewriting, especially rewrite-retrieve-read, <br>
        ... know how to split a question into sub-questions (query decomposition), <br>
        ... know how to implement chains the LangChain way.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [LangChain Docs: Build a RAG agent](https://docs.langchain.com/oss/python/langchain/rag)
* [LangChain API Reference](https://reference.langchain.com/python/)
* [LangChain AI (2024) Cookbook](https://github.com/langchain-ai/langchain/blob/master/cookbook/rewrite.ipynb?ref=blog.langchain.dev)

## Where We Are

* **5.04:** reranking. Better chunks *after* the search.
* **Now:** **query rewriting**. A better question *before* the search.

![RAG extensions](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/rag_extensions.png)

(Source: [Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981))

Query rewriting is one of many ways to improve RAG ([Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981)).

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu python-dotenv

In [ ]:
import os
import re
import urllib.request

from IPython.display import display, Markdown

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# downloads the documents, if they are not here yet
BASE_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in [
    "documents/Game_of_Thrones.pdf",
    "documents/How_I_Met_Your_Mother.pdf",
    "markdown_documents/Game_of_Thrones.md",
    "markdown_documents/How_I_Met_Your_Mother.md",
]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + path, path)

### Connect to the AI-Gateway

The same setup as in 4.03: your own computer, connected to the university network or the VPN, and your AI-Gateway key.

We use two models on the gateway:

* a **chat model** that writes the answers (the generator), and
* an **embedding model** that turns texts into vectors (the retriever).

**Good to know:** the gateway allows about 10 requests per minute per model. If a cell pauses, the client is waiting for the next minute and then continues by itself.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads the .env file: searches this folder first, then the folders above (the repo folder)
API_KEY = os.getenv("AI_GATEWAY_KEY")
AI_GATEWAY_BASE = os.getenv("AI_GATEWAY_BASE")  # address of the university's AI-Gateway
if not API_KEY or not AI_GATEWAY_BASE:
    raise ValueError("AI_GATEWAY_KEY or AI_GATEWAY_BASE missing. Copy .env.example to .env in the repo folder and paste your key there.")

GENERATOR_NAME = "gwdg.qwen3.8-27b-no-think"  # chat model: writes the answers
RETRIEVER_NAME = "gwdg.qwen3-embedding-4b"    # embedding model: turns texts into vectors

The key comes from the `.env` file in the repo folder (set up in 4.03). If the cell says that no key was found, copy `.env.example` to `.env` and paste your key behind `AI_GATEWAY_KEY=`.

## Loading the Documents

The Markdown files from 5.02, with the links removed.

In [ ]:
markdown_documents_path = "markdown_documents"

In [ ]:
def remove_markdown_links(text: str) -> str:
    # 1. Remove inline links: [text](url)
    text = re.sub(r'\[([^\]]*)\]\([^)]+\)', r'\1', text)

    # 2. Remove reference-style links: [text][ref]
    text = re.sub(r'\[([^\]]*)\]\[[^\]]*\]', r'\1', text)

    # 3. Remove malformed/broken bracket groups like [[][1][]]
    text = re.sub(r'\[+\s*\]+', '', text)

    # 4. Remove reference definitions like: [1]: https://...
    text = re.sub(r'^\s*\[[^\]]+\]:.*$', '', text, flags=re.MULTILINE)

    return text

In [ ]:
markdown_documents = sorted(os.listdir(markdown_documents_path))

md_files = []

for markdown_document in markdown_documents:
    markdown_document_path = os.path.join(markdown_documents_path, markdown_document)
    with open(markdown_document_path, encoding="utf-8") as file:
        md_files.append([markdown_document, remove_markdown_links(file.read())])

## Part 1: A Distracted Question

**In short: an unrelated part of the question can pull the retriever to the wrong chunks.**

### The vector store: build it once, then load it

Embedding all chunks takes about a minute and costs about 150,000 tokens of your gateway budget. So we do it only once:

* If you ran **5.03** in this folder, the index is saved in the folder `faiss_index`, and we just load it.
* Otherwise, we build it now and save it for the next notebooks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
all_splits = text_splitter.create_documents(
    texts=[x[1] for x in md_files],
    metadatas=[{"source": x[0]} for x in md_files],
)

retriever_model = OpenAIEmbeddings(
    model=RETRIEVER_NAME,
    base_url=AI_GATEWAY_BASE,
    api_key=API_KEY,
    check_embedding_ctx_length=False,  # send plain text; the default would first split it with OpenAI's own tokenizer
)

if os.path.exists("faiss_index"):
    db = FAISS.load_local("faiss_index", retriever_model, allow_dangerous_deserialization=True)  # our own file, so loading is safe
else:
    db = FAISS.from_documents(all_splits, embedding=retriever_model)
    db.save_local("faiss_index")
print(db.index.ntotal, "chunks in the vector store")

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
llm = ChatOpenAI(model=GENERATOR_NAME, base_url=AI_GATEWAY_BASE, api_key=API_KEY, temperature=0)

In [ ]:
prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)
print(prompt.template)

The chain from 5.03. The retriever returns a list of documents; `format_docs()` joins their texts into one context string.

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
chain.invoke("I loooove Naomi Watts! Who plays Daenerys Targaryen?")

Let's look at what the retriever found for this question:

In [ ]:
for doc in retriever.invoke("I loooove Naomi Watts! Who plays Daenerys Targaryen?"):
    print(doc.metadata["source"], "|", doc.page_content[:150].replace("\n", " "), "...")

The question contains a distraction: Naomi Watts has nothing to do with Daenerys Targaryen. The retriever embeds the *whole* question, so the distraction can pull it towards the wrong chunks. How can we prevent this?

## Part 2: Rewrite-Retrieve-Read

**In short: let an LLM turn the question into a better search query first.**

1. **Rewrite:** an LLM turns the question into a search query.
2. **Retrieve:** the retriever searches with this query.
3. **Read:** the LLM answers the *original* question from the retrieved chunks.

Rewrite-Retrieve-Read (RRR) was introduced by [Ma et al. (2023)](https://aclanthology.org/2023.emnlp-main.322/). There are many other ways to transform queries; this [LangChain blog post](https://blog.langchain.dev/query-transformations/) explains them well.

First, the rewriter on its own. `_parse()` takes the search query out of the model's answer:

In [ ]:
template = """Provide a better search query for web search engine to answer the given question, end the queries with ’**’. Question: {x} Answer:"""
rewrite_prompt = ChatPromptTemplate.from_template(template)


def _parse(text):
    """Takes the query between the ** markers. If the model did not use them, it takes the whole answer."""
    text = text.replace("’**’", "**")
    parts = [part.strip() for part in text.split("**") if part.strip()]
    return parts[-1] if parts else text.strip()


rewriter = rewrite_prompt | llm | StrOutputParser() | _parse
rewriter.invoke({"x": "I loooove Naomi Watts! Who plays Daenerys Targaryen?"})

### The updated chain

The rewriter goes in front of the retriever. The prompt still gets the original question.

In [ ]:
rewrite_retrieve_read_chain = (
    {
        "context": {"x": RunnablePassthrough()} | rewriter | retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)
rewrite_retrieve_read_chain.invoke("I loooove Naomi Watts! Who plays Daenerys Targaryen?")

## Part 3: Query Decomposition

**In short: a question with several parts becomes several search queries.**

The retriever returns the 3 chunks that are closest to the *whole* question. What happens if the question asks about two things at once? Let's look at what the retriever finds:

In [ ]:
QUESTION = "Who plays Jon Snow and who plays Barney Stinson?"

for doc in retriever.invoke(QUESTION):
    print(doc.metadata["source"], "|", doc.page_content[:150].replace("\n", " "), "...")

In [ ]:
chain.invoke(QUESTION)

**Query decomposition** splits the question into simple sub-questions ([LangChain blog](https://blog.langchain.dev/query-transformations/)):

1. **Decompose:** an LLM splits the question into sub-questions.
2. **Retrieve:** the retriever searches once for each sub-question, and we collect all chunks in one list.
3. **Read:** the LLM answers the *original* question from all chunks together.

### Your turn

1. Write `decompose_template`: a prompt that splits the question `{x}` into sub-questions, one per line, and nothing else. `_split_lines()` then turns the answer into a Python list.
2. Complete `retrieve_decomposed()`: retrieve the chunks for every sub-question and collect them in `docs`, without duplicates.
3. Build `decomposition_chain`: like `rewrite_retrieve_read_chain`, but with `RunnableLambda(retrieve_decomposed)` in front of `format_docs`. `RunnableLambda` turns a normal Python function into a block for the chain.

Then compare the chunks and the answer with the chain above.

In [ ]:
# TODO 1: write a prompt that splits the question {x} into sub-questions, one per line
decompose_template = """"""

def _split_lines(text):
    """Turns the model's answer into a list with one sub-question per line, without list markers such as "1." or "-"."""
    lines = [line.strip().lstrip("-*0123456789.) ").strip() for line in text.split("\n")]
    return [line for line in lines if line]


decomposer = ChatPromptTemplate.from_template(decompose_template) | llm | StrOutputParser() | _split_lines
decomposer.invoke({"x": QUESTION})

In [ ]:
# TODO 2: retrieve the chunks for every sub-question and collect them in docs, without duplicates
def retrieve_decomposed(question):
    sub_questions = decomposer.invoke({"x": question})
    docs = []

    # write your code here

    return docs


for doc in retrieve_decomposed(QUESTION):
    print(doc.metadata["source"], "|", doc.page_content[:150].replace("\n", " "), "...")

In [ ]:
# TODO 3: the chain with RunnableLambda(retrieve_decomposed) in front of format_docs
decomposition_chain = None  # replace None with your chain

decomposition_chain.invoke(QUESTION)

## What You Learned

* The retriever embeds the **whole** question, so distractions in the question lead to wrong chunks.
* **Rewrite-Retrieve-Read:** an LLM first rewrites the question into a clean search query; the answer still uses the original question.
* In LangChain, the rewriter is just one more block in the chain, in front of the retriever.
* **Query decomposition:** a question with several parts becomes several sub-questions. The retriever searches for each one, and the LLM gets all chunks together.

## Want to learn more about RAG with LangChain?

* [LangChain RAG from scratch](https://github.com/langchain-ai/rag-from-scratch)
* [LangChain Docs: Build a RAG agent](https://docs.langchain.com/oss/python/langchain/rag)